> 在线讲义：[第 14 章 · SwiGLU：门控为什么有用，8/3 怎么保住参数量](https://wikieden.github.io/beyond-transformer/lessons/14-swiglu.html) · 本书《超越 Transformer》

# 第 14 章 · SwiGLU：门控为什么有用，8/3 怎么保住参数量

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/14-swiglu.html) · 仓库里 `beyond-transformer/notes/lessons/14-swiglu.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 激活 | x·σ(x) | `relu`、`gelu`、`silu` | SiLU(1) = 0.7311 |
| 第二步 门 × 内容 | 式 (6) | `swiglu()` | 输出 [3.404, 0.1192] |
| 第三步 参数 | 3·d·d_ff | `swiglu_dff(d)`、`params(...)` | d = 768 时 2048、参数相等 |

Swish（SiLU）和 ReLU、GELU 长什么样；一个小例子看门怎么“关掉”一部分内容。
a1 的 positionwise_feedforward（SwiGLU 模块）和 swiglu_ablation 的训练对比留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `relu`**：小工具。

In [ ]:
def relu(x):
    return np.maximum(x, 0)

**函数 `silu`**：Swish₁ = SiLU：x·σ(x)。

In [ ]:
def silu(x):
    """Swish₁ = SiLU：x·σ(x)。"""
    return x / (1 + np.exp(-x))

**函数 `gelu`**：小工具。

In [ ]:
def gelu(x):
    return 0.5 * x * (1 + np.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)))

In [ ]:
X = np.array([1.0, -1.0])
W = np.array([[2.0, 0.0, -1.0], [0.0, 2.0, 1.0]])      # 门：d = 2 → 3
V = np.array([[1.0, 1.0, 1.0], [-1.0, 1.0, 0.5]])      # 内容：d = 2 → 3
W2 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, -1.0]])   # 3 → 2

**函数 `swiglu`**：小工具。

In [ ]:
def swiglu(x=X, w=W, v=V, w2=W2):
    gate = silu(x @ w)
    content = x @ v
    return gate, content, (gate * content) @ w2

**函数 `params`**：小工具。

In [ ]:
def params(d: int, d_ff: int, gated: bool) -> int:
    return (3 if gated else 2) * d * d_ff

**函数 `swiglu_dff`**：8/3·d，再向上取到 64 的倍数（让矩阵乘法对齐 GPU 的 tensor core）。

In [ ]:
def swiglu_dff(d: int, multiple: int = 64) -> int:
    """8/3·d，再向上取到 64 的倍数（让矩阵乘法对齐 GPU 的 tensor core）。"""
    return int(math.ceil(8 * d / 3 / multiple) * multiple)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 三种激活")
for x in (-3.0, -1.0, 0.0, 1.0, 3.0):
    print(f"  x = {x:+.0f}：ReLU {relu(x):.4f}，GELU {gelu(x):.4f}，SiLU {silu(x):.4f}")
print("\n第二步 门 × 内容")
g, c, y = swiglu()
print(f"  xW = {(X @ W).tolist()} → 门 SiLU(xW) = {np.round(g, 4).tolist()}")
print(f"  内容 xV = {c.tolist()}；门 ⊙ 内容 = {(np.round(g * c, 4) + 0.0).tolist()}；输出 = {np.round(y, 4).tolist()}")

In [ ]:
print("\n第三步 参数预算")
for d in (512, 768, 4096):
    dff = swiglu_dff(d)
    print(f"  d = {d}：普通 FFN（d_ff = 4d）{params(d, 4 * d, False):,}；SwiGLU（d_ff = {dff}）{params(d, dff, True):,}")

## 3. 代码里才看得到的

- **向上取整**：这里用 ceil，所以 d = 512 时 SwiGLU 比普通 FFN 多 3% 参数；有的实现向下取或按 256 取，几个百分点的差别。
- **没有偏置**：GLU 论文和 LLaMA 都去掉了 bias（L66–L67），这里的矩阵乘法也不带 bias。

## 4. 自己改一改

- 把 `silu` 换成 `relu`（ReGLU），正文例子的输出变成多少？
- 把 `swiglu_dff` 的倍数改成 256，d = 4096 时得到多少？是不是 LLaMA 的 11008？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> SiLU 是 x 乘 sigmoid，平滑的 ReLU，负半边略低一点。
> 一路当门一路当内容，逐元素一乘，门关的维度就被压掉；为什么好用，归功于神的恩典。
> 三个矩阵比两个，中间层乘三分之八，参数不变；再取整到 64 的倍数。